## Feature Engineering

### By:
Karen Ninco

### Date:
2026-08-21

### Description:

# Requerimiento

Realizar el proceso de Feature Engineering para limpieza, transformación y modificación de los datos para que puedan ser usados para entrenar un modelo para resolver un problema. Hacerlo en un nuevo branch de git (Usar [Gitflow](https://joserzapata.github.io/courses/ciencia-datos-en-produccion/control-versiones/branching-model/)) y Crear un notebook  para la creación de los pipelines de scikit-learn

> Tomar como ejemplo los pasos de: <https://joserzapata.github.io/post/ciencia-datos-proyecto-python/4-feat_eng/>

En este proceso para cada tipo de datos se incluye :

1. **Limpieza de datos**:
  - Eliminar registros datos duplicados (disminuir el numero de datos)
  - Corregir o eliminar valores atípicos (opcional).
  - Los valores atípicos pueden separarse del dataset dependiendo del problema del proyecto (por ejemplo, detección de anomalías).
  - Completar los valores faltantes (por ejemplo, con cero, media, mediana …) o eliminar las filas (o columnas).
2. Selección de atributos (**Feature Selection**) (opcional):
  - Descartar los atributos que no proporcionan información útil para el proyecto.
  - Eliminar registros duplicados (al eliminar atributos pueden quedar registros iguales)
3. Ingeniería de atributos (**Feature Engineering**), cuando sea apropiado:
  - Discretizar las atributos continuas.
  - (opcional) Descomponer en partes los atributos (p. Ej., Categóricas, fecha / hora, etc.).
  - (opcional) Agregar transformaciones prometedoras de las atributos, por ejemplo:
    - log(x)
    - sqrt(x)
    - x^2
    - etc
  - Aplicar funciones a los datos para agregar nuevos atributos.
4. Escalado de atributos (**Feature Scaling**):
  - estandarizar
  - normalizar
  - etc
5. Encoding
  - Encode variables categóricas, texto y las que sean necesarias para poder ser usadas para el modelamiento

Crear todas estas transformaciones usando transformadores y pipelines de scikit-learn
-  https://scikit-learn.org/stable/modules/preprocessing.html
-  https://scikit-learn.org/stable/modules/compose.html#

> Puede utilizar las librerías o herramientas que considere para resolver la tarea

# Entregables

Notebook con la descripción y creación de los pipelines de scikit-learn.

Se debe realizar un Pull request para ingresar el notebook a la rama`main` para esto debe tener mínimo 1 revisiones de otras personas del Curso y que pase los check del CI/CD.


## 📚 Import libraries

In [86]:
# base libraries for data science
from pathlib import Path

import pandas as pd
import sklearn as sk
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

print("Pandas version: ", pd.__version__)
print("sklearn version: ", sk.__version__)

Pandas version:  3.0.5
sklearn version:  1.9.0


## 💾 Load data

In [87]:
DATA_DIR = Path.cwd().resolve().parents[1] / "data"
corazon_df = pd.read_parquet(
    DATA_DIR / "02_intermediate/corazon_clean.parquet", engine="pyarrow"
)

### Diagnóstico: tipo de orden de las variables categóricas

In [88]:
for col in corazon_df.select_dtypes(include=["category"]).columns:
    print(
        col,
        "-> ordered:",
        corazon_df[col].cat.ordered,
        "| categories:",
        list(corazon_df[col].cat.categories),
    )

sex -> ordered: False | categories: ['Female', 'Male']
chest_pain -> ordered: False | categories: ['asymptomatic', 'nonanginal', 'nontypical', 'typical']
rest_ecg -> ordered: False | categories: ['ST-T wave abnormality', 'left ventricular hypertrophy', 'normal']
slope -> ordered: True | categories: ['1', '2', '3']
thal -> ordered: False | categories: ['fixed', 'normal', 'reversable']


## Contexto del dataset

- Variable objetivo: `disease` (booleana, con 111 valores nulos que deben eliminarse antes de dividir train/test, ya que no se pueden usar para entrenar un modelo supervisado)
- Columnas numéricas: age, rest_bp, chol, max_hr, old_peak, ca
- Columnas categóricas nominales (sin orden): sex, chest_pain, rest_ecg, thal
- Columna categórica ordinal (con orden): slope (orden real: 1 < 2 < 3, ya confirmado con `corazon_df["slope"].cat.ordered = True`)
- Columnas booleanas: fbs, exang (ya están en 0/1, NO necesitan OneHotEncoder ni OrdinalEncoder, solo SimpleImputer)

Nota: `fbs` y `rest_bp` se eliminan más adelante en Feature Selection por su baja asociación con la variable objetivo, por lo que no aparecen en el dataset final.

## 👷 Limpieza de Datos

Se parte de las 14 columnas del dataset ya preprocesado (`corazon_clean.parquet`). Sobre este conjunto se aplicará limpieza de duplicados y, más adelante en la sección de Feature Selection, se descartarán las columnas con baja asociación con la variable objetivo.

In [89]:
selected_features = [
    "age",
    "sex",
    "chest_pain",
    "rest_bp",
    "chol",
    "fbs",
    "rest_ecg",
    "max_hr",
    "exang",
    "old_peak",
    "slope",
    "ca",
    "thal",
    "disease",
]
corazon_features = corazon_df[selected_features].copy()
corazon_features.info()

<class 'pandas.DataFrame'>
RangeIndex: 3030 entries, 0 to 3029
Data columns (total 14 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   age         2998 non-null   float64 
 1   sex         2966 non-null   category
 2   chest_pain  2944 non-null   category
 3   rest_bp     2947 non-null   float64 
 4   chol        2943 non-null   float64 
 5   fbs         2933 non-null   boolean 
 6   rest_ecg    2832 non-null   category
 7   max_hr      2858 non-null   Int16   
 8   exang       2877 non-null   boolean 
 9   old_peak    2878 non-null   float64 
 10  slope       2878 non-null   category
 11  ca          2867 non-null   Int8    
 12  thal        2900 non-null   category
 13  disease     2919 non-null   boolean 
dtypes: Int16(1), Int8(1), boolean(3), category(5), float64(4)
memory usage: 142.5 KB


### Missing values

In [90]:
corazon_features.isna().sum()

age            32
sex            64
chest_pain     86
rest_bp        83
chol           87
fbs            97
rest_ecg      198
max_hr        172
exang         153
old_peak      152
slope         152
ca            163
thal          130
disease       111
dtype: int64

Antes de dividir en train/test, se eliminan las filas donde la variable objetivo `disease` es nula, ya que estos registros no pueden usarse para entrenar un modelo supervisado.

In [91]:
corazon_features = corazon_features.dropna(subset=["disease"])
corazon_features.info()

<class 'pandas.DataFrame'>
Index: 2919 entries, 0 to 3029
Data columns (total 14 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   age         2902 non-null   float64 
 1   sex         2870 non-null   category
 2   chest_pain  2848 non-null   category
 3   rest_bp     2851 non-null   float64 
 4   chol        2847 non-null   float64 
 5   fbs         2837 non-null   boolean 
 6   rest_ecg    2803 non-null   category
 7   max_hr      2829 non-null   Int16   
 8   exang       2848 non-null   boolean 
 9   old_peak    2849 non-null   float64 
 10  slope       2849 non-null   category
 11  ca          2838 non-null   Int8    
 12  thal        2872 non-null   category
 13  disease     2919 non-null   boolean 
dtypes: Int16(1), Int8(1), boolean(3), category(5), float64(4)
memory usage: 160.0 KB


### Duplicated data

In [92]:
duplicate_rows = corazon_features.duplicated().sum()
print("Number of duplicate rows: ", duplicate_rows)

Number of duplicate rows:  2439


In [93]:
corazon_features.sample(10, random_state=42)

,age,sex,chest_pain,rest_bp,chol,fbs,rest_ecg,max_hr,exang,old_peak,slope,ca,thal,disease
2548,65.0,Male,typical,138.0,282.0,True,left ventricular hypertrophy,174,False,1.4,2,1,normal,True
2581,58.0,Male,asymptomatic,125.0,300.0,False,left ventricular hypertrophy,171,False,0.0,1,2,reversable,True
2470,51.0,Male,nonanginal,110.0,175.0,False,normal,123,False,0.6,1,0,normal,False
794,54.0,Male,nontypical,192.0,283.0,False,left ventricular hypertrophy,195,False,0.0,1,1,reversable,True
1753,49.0,Female,nontypical,134.0,271.0,False,normal,162,False,0.0,2,0,normal,False
196,69.0,Male,typical,160.0,234.0,True,left ventricular hypertrophy,131,False,0.1,2,1,normal,False
1654,51.0,Male,nonanginal,125.0,245.0,True,left ventricular hypertrophy,166,False,2.4,2,0,normal,False
1381,45.0,Female,nontypical,112.0,160.0,False,normal,138,False,0.0,2,0,normal,False
2584,46.0,Male,nontypical,101.0,197.0,True,normal,156,False,0.0,1,0,reversable,False
2184,54.0,Female,nonanginal,135.0,304.0,True,normal,170,False,0.0,1,0,normal,False


In [94]:
corazon_features = corazon_features.drop_duplicates()
corazon_features.info()

<class 'pandas.DataFrame'>
Index: 480 entries, 0 to 1496
Data columns (total 14 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   age         463 non-null    float64 
 1   sex         431 non-null    category
 2   chest_pain  409 non-null    category
 3   rest_bp     412 non-null    float64 
 4   chol        408 non-null    float64 
 5   fbs         398 non-null    boolean 
 6   rest_ecg    364 non-null    category
 7   max_hr      390 non-null    Int16   
 8   exang       409 non-null    boolean 
 9   old_peak    410 non-null    float64 
 10  slope       410 non-null    category
 11  ca          433 non-null    Int8    
 12  thal        448 non-null    category
 13  disease     480 non-null    boolean 
dtypes: Int16(1), Int8(1), boolean(3), category(5), float64(4)
memory usage: 26.6 KB


## Feature Selection

En el análisis bivariable (03b) se encontró que `fbs` y `rest_bp` muestran muy poca o ninguna asociación con la variable objetivo `disease` (diferencias de menos de 3 puntos porcentuales entre categorías). Con base en ese hallazgo, se descartan estas dos columnas antes de construir los pipelines, ya que aportarían poca información útil al modelo.

In [95]:
corazon_features = corazon_features.drop(columns=["fbs", "rest_bp"])
corazon_features.info()

<class 'pandas.DataFrame'>
Index: 480 entries, 0 to 1496
Data columns (total 12 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   age         463 non-null    float64 
 1   sex         431 non-null    category
 2   chest_pain  409 non-null    category
 3   chol        408 non-null    float64 
 4   rest_ecg    364 non-null    category
 5   max_hr      390 non-null    Int16   
 6   exang       409 non-null    boolean 
 7   old_peak    410 non-null    float64 
 8   slope       410 non-null    category
 9   ca          433 non-null    Int8    
 10  thal        448 non-null    category
 11  disease     480 non-null    boolean 
dtypes: Int16(1), Int8(1), boolean(2), category(5), float64(3)
memory usage: 21.9 KB


### Revisión de duplicados tras Feature Selection

Al eliminar las columnas `fbs` y `rest_bp`, es posible que registros que antes eran distintos (por diferir solo en esas columnas) ahora resulten idénticos. Se revisa nuevamente la cantidad de duplicados sobre el conjunto reducido de columnas.

In [96]:
duplicate_rows_after_selection = corazon_features.duplicated().sum()
print(
    "Number of new duplicate rows after feature selection: ",
    duplicate_rows_after_selection,
)

if duplicate_rows_after_selection > 0:
    corazon_features = corazon_features.drop_duplicates()

corazon_features.info()

Number of new duplicate rows after feature selection:  7
<class 'pandas.DataFrame'>
Index: 473 entries, 0 to 1496
Data columns (total 12 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   age         456 non-null    float64 
 1   sex         424 non-null    category
 2   chest_pain  402 non-null    category
 3   chol        401 non-null    float64 
 4   rest_ecg    357 non-null    category
 5   max_hr      383 non-null    Int16   
 6   exang       402 non-null    boolean 
 7   old_peak    403 non-null    float64 
 8   slope       403 non-null    category
 9   ca          426 non-null    Int8    
 10  thal        441 non-null    category
 11  disease     473 non-null    boolean 
dtypes: Int16(1), Int8(1), boolean(2), category(5), float64(3)
memory usage: 21.6 KB


## 👨‍🏭 Feature Engineering

### Encode target variable

In [97]:
corazon_features["disease"] = corazon_features["disease"].astype("int")
# True = 1, False = 0
corazon_features.sample(5)

,age,sex,chest_pain,chol,rest_ecg,max_hr,exang,old_peak,slope,ca,thal,disease
22,58.0,Male,nontypical,284.0,left ventricular hypertrophy,160,False,1.8,2,0,normal,1
1477,42.0,Male,asymptomatic,315.0,NaN,<NA>,<NA>,NaN,NaN,<NA>,NaN,1
191,51.0,Male,asymptomatic,298.0,normal,122,True,4.2,2,3,reversable,1
1492,57.0,Male,asymptomatic,335.0,NaN,<NA>,<NA>,NaN,NaN,<NA>,NaN,1
180,48.0,Male,asymptomatic,274.0,left ventricular hypertrophy,166,False,0.5,2,0,reversable,1


### Discretización de atributos continuos

Se consideró discretizar `chol` (colesterol) y `rest_bp` (presión arterial en reposo) en categorías clínicas estándar (por ejemplo: normal / límite alto / alto), ya que existen umbrales médicos reconocidos para ambas variables. Sin embargo, se decidió NO aplicar esta discretización porque no se contaba con una fuente confiable y verificada de esos umbrales exactos en el contexto de este ejercicio, y documentar valores de corte incorrectos sería peor que no discretizar.

Respecto a `age`, también se evaluó discretizarla en rangos (por ejemplo, por décadas), pero se descartó porque en el análisis exploratorio (03c) no se encontró evidencia de un umbral o punto de quiebre claro en la relación entre edad y la enfermedad — la relación se observa gradual y consistente, no abrupta. Discretizarla habría sido una decisión arbitraria sin respaldo en los datos, y además le quitaría a los modelos basados en árboles (que se probarán en la siguiente actividad) la posibilidad de encontrar automáticamente el punto de corte óptimo.

### Feature Scaling y Encoding — pipelines de transformación

A continuación se construyen los pipelines de scikit-learn: el pipeline numérico aplica escalado (`StandardScaler`, Punto 4), y los pipelines categóricos aplican codificación (`OneHotEncoder` y `OrdinalEncoder`, Punto 5). Se agrupan en una sola celda porque se combinan luego en un mismo `ColumnTransformer`.

In [98]:
cols_numeric = ["age", "chol", "max_hr", "old_peak", "ca"]
cols_boolean = ["exang"]
cols_categoric = ["sex", "chest_pain", "rest_ecg", "thal"]
cols_categoric_ord = ["slope"]

numeric_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

boolean_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
    ]
)

categorical_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder()),
    ]
)

categorical_ord_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OrdinalEncoder()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipe, cols_numeric),
        ("boolean", boolean_pipe, cols_boolean),
        ("categoric", categorical_pipe, cols_categoric),
        ("categoric ordinales", categorical_ord_pipe, cols_categoric_ord),
    ]
)
preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('boolean', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``f

- **Pipeline numérico** (`numeric_pipe`): imputa valores faltantes en las columnas numéricas usando la mediana, ya que es una medida robusta frente a valores atípicos, y luego aplica `StandardScaler` para estandarizar las variables (media 0, desviación estándar 1). Esto pone a todas las columnas numéricas en la misma escala, lo cual es importante para varios algoritmos de machine learning (por ejemplo regresión logística, SVM o KNN) que son sensibles a la magnitud de los valores de entrada.
- **Pipeline booleano** (`boolean_pipe`): imputa los valores faltantes de la columna booleana (`exang`) usando la moda (valor más frecuente), sin necesidad de codificación adicional ya que esta columna ya está en formato 0/1.
- **Pipeline categórico nominal** (`categorical_pipe`): imputa con la moda y aplica `OneHotEncoder` para crear variables dummy, adecuado para variables sin orden entre sus categorías (`sex`, `chest_pain`, `rest_ecg`, `thal`).
- **Pipeline categórico ordinal** (`categorical_ord_pipe`): imputa con la moda y aplica `OrdinalEncoder`, preservando el orden lógico de la variable `slope` (1 < 2 < 3).
- **ColumnTransformer** (`preprocessor`): combina los cuatro pipelines anteriores, aplicando cada uno únicamente a su conjunto de columnas correspondiente, para producir el conjunto final de features listas para el modelo.

### Train / Test split

In [99]:
X_features = corazon_features.drop("disease", axis="columns")
Y_target = corazon_features["disease"]

# 80% train, 20% test
x_train, x_test, y_train, y_test = train_test_split(
    X_features, Y_target, test_size=0.2, stratify=Y_target, random_state=42
)
x_train.shape, y_train.shape

((378, 11), (378,))

In [100]:
x_test.shape, y_test.shape

((95, 11), (95,))

### Preprocessing pipeline

In [101]:
transformed_data = preprocessor.fit(x_train)
feature_names = preprocessor.get_feature_names_out()

x_train_transformed = preprocessor.transform(x_train)
x_train_transformed = pd.DataFrame(x_train_transformed, columns=feature_names)
x_train_transformed.info()

<class 'pandas.DataFrame'>
RangeIndex: 378 entries, 0 to 377
Data columns (total 19 columns):
 #   Column                                            Non-Null Count  Dtype  
---  ------                                            --------------  -----  
 0   numeric__age                                      378 non-null    float64
 1   numeric__chol                                     378 non-null    float64
 2   numeric__max_hr                                   378 non-null    float64
 3   numeric__old_peak                                 378 non-null    float64
 4   numeric__ca                                       378 non-null    float64
 5   boolean__exang                                    378 non-null    float64
 6   categoric__sex_Female                             378 non-null    float64
 7   categoric__sex_Male                               378 non-null    float64
 8   categoric__chest_pain_asymptomatic                378 non-null    float64
 9   categoric__chest_pain_nonanginal

In [102]:
x_train_transformed.head()

,numeric__age,numeric__chol,numeric__max_hr,numeric__old_peak,numeric__ca,boolean__exang,categoric__sex_Female,categoric__sex_Male,categoric__chest_pain_asymptomatic,categoric__chest_pain_nonanginal,categoric__chest_pain_nontypical,categoric__chest_pain_typical,categoric__rest_ecg_ST-T wave abnormality,categoric__rest_ecg_left ventricular hypertrophy,categoric__rest_ecg_normal,categoric__thal_fixed,categoric__thal_normal,categoric__thal_reversable,categoric ordinales__slope
0,1.087241,1.186115,-0.903839,0.687944,-0.667695,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0
1,0.497604,-1.455219,-2.893569,-0.026950,1.546241,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0
2,0.851386,0.371869,0.503531,2.296456,1.546241,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,2.0
3,-1.507163,-1.375780,0.115291,-0.205674,-0.667695,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
4,0.379677,-1.038166,0.115291,-0.205674,-0.667695,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0


### Transformar datos de test

Se aplica el `preprocessor` ya ajustado (con `fit` sobre `x_train`) para transformar `x_test`, sin volver a ajustarlo. Esto evita fuga de información: el conjunto de prueba nunca participa en el cálculo de la mediana, la moda o los parámetros del `StandardScaler`.

In [103]:
x_test_transformed = preprocessor.transform(x_test)
x_test_transformed = pd.DataFrame(x_test_transformed, columns=feature_names)
x_test_transformed.info()

<class 'pandas.DataFrame'>
RangeIndex: 95 entries, 0 to 94
Data columns (total 19 columns):
 #   Column                                            Non-Null Count  Dtype  
---  ------                                            --------------  -----  
 0   numeric__age                                      95 non-null     float64
 1   numeric__chol                                     95 non-null     float64
 2   numeric__max_hr                                   95 non-null     float64
 3   numeric__old_peak                                 95 non-null     float64
 4   numeric__ca                                       95 non-null     float64
 5   boolean__exang                                    95 non-null     float64
 6   categoric__sex_Female                             95 non-null     float64
 7   categoric__sex_Male                               95 non-null     float64
 8   categoric__chest_pain_asymptomatic                95 non-null     float64
 9   categoric__chest_pain_nonanginal  

In [104]:
x_test_transformed.head()

,numeric__age,numeric__chol,numeric__max_hr,numeric__old_peak,numeric__ca,boolean__exang,categoric__sex_Female,categoric__sex_Male,categoric__chest_pain_asymptomatic,categoric__chest_pain_nonanginal,categoric__chest_pain_nontypical,categoric__chest_pain_typical,categoric__rest_ecg_ST-T wave abnormality,categoric__rest_ecg_left ventricular hypertrophy,categoric__rest_ecg_normal,categoric__thal_fixed,categoric__thal_normal,categoric__thal_reversable,categoric ordinales__slope
0,-0.209961,-0.065043,1.134421,-0.920568,2.653209,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
1,0.851386,-0.084903,0.115291,0.330497,-0.667695,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
2,0.733459,-0.084903,-0.175889,-0.026950,-0.667695,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0
3,0.497604,-0.084903,0.115291,-0.205674,1.546241,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0
4,1.912734,-1.991430,-1.195019,0.509220,-0.667695,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0


## 📊 Analysis of Results and Conclusions 

### Resultados obtenidos

Se construyó un pipeline de preprocesamiento con scikit-learn (`ColumnTransformer`) que combina 4 sub-pipelines según el tipo de variable: numéricas (imputación con mediana + `StandardScaler`), booleana (imputación con moda), categóricas nominales (imputación con moda + `OneHotEncoder`) y la categórica ordinal `slope` (imputación con moda + `OrdinalEncoder`, respetando su orden real 1 < 2 < 3, confirmado antes de construir el pipeline). El pipeline se ajustó (`fit`) únicamente sobre los datos de entrenamiento (`x_train`), y se aplicó (`transform`) tanto a `x_train` como a `x_test` para producir un dataset final de 19 columnas numéricas, sin ningún valor nulo restante.

### Hallazgo importante: alta proporción de datos duplicados

Antes de dividir en train/test, se eliminaron las 111 filas sin valor en la variable objetivo (`disease`), quedando 2919 filas. Al revisar duplicados exactos sobre esas 2919 filas, se encontró que **2439 (83.5%) son duplicados**, dejando **480 filas únicas**.

Tras eliminar las columnas `fbs` y `rest_bp` en Feature Selection, se revisó nuevamente la existencia de duplicados (ya que registros antes distintos pueden volverse idénticos al quitar columnas), encontrando **7 duplicados adicionales**, para un total de **473 filas únicas** en el dataset final.

Esto es consistente con lo observado en el análisis univariable (03a), donde se había detectado un nivel similar de duplicados (2462 de 3030 filas, antes de filtrar `disease`; aquí, 2439 de 2919, después del filtro). Al no existir una columna de identificación de paciente, no se puede confirmar con certeza si son registros repetidos o pacientes distintos con características similares.

La consecuencia directa es que, después de dividir en train/test (80/20, `random_state=42`) sobre las 473 filas únicas, el conjunto de entrenamiento final quedó en **378 filas** y el de prueba en **95 filas**. Esta es una cantidad de datos considerablemente pequeña para entrenar un modelo de machine learning de forma confiable, sobre todo después de que el encoding de variables categóricas aumentó el número de columnas de 11 a 19.


## 💡 Proposals and Ideas

Dado el hallazgo de la baja cantidad de datos únicos disponibles (473 filas), se propone:

- Evaluar en las siguientes etapas (Modelo Base, Selección del mejor modelo) si es mejor usar el dataset completo con duplicados (2919 filas) en lugar de eliminarlos, dado que no hay certeza de que sean registros erróneos y no pacientes distintos con características similares — esta decisión debe tomarse con cuidado, ya que también afecta la validez estadística del modelo si se entrena con filas repetidas.
- Si se decide mantener la eliminación de duplicados, considerar técnicas de validación cruzada con más folds para aprovechar mejor las 473 filas disponibles, y ser cautelosos con el riesgo de overfitting dado el tamaño reducido de la muestra.
- Documentar esta limitación claramente en cualquier reporte o presentación del proyecto, ya que afecta la confiabilidad de las conclusiones del modelo final.

**Handling Missing Data**

Actualmente se usan estrategias simples de imputación (mediana para numéricas, moda para categóricas y booleana). Como trabajo futuro se podrían explorar estrategias más sofisticadas, como `KNNImputer` o `IterativeImputer`, que aprovechan la relación entre variables para imputar valores faltantes de forma más precisa.

**Feature Engineering**

Se podrían explorar nuevas variables derivadas a partir de las existentes, de forma similar a como la guía combina `sibsp` y `parch` para crear el tamaño de familia en el dataset Titanic. Por ejemplo:
- Una variable que combine `age` y `max_hr` para reflejar la reserva de frecuencia cardíaca del paciente.
- Una variable que combine `old_peak` y `slope`, ya que ambas describen el comportamiento del segmento ST durante el ejercicio.
- Un score de riesgo compuesto que agrupe variables categóricas asociadas a mayor riesgo cardíaco (`chest_pain`, `exang`).

Estas ideas deberían evaluarse con el equipo clínico/dominio antes de implementarse, para validar su relevancia real.
